In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    MultiHeadAttention,
    LayerNormalization,
    Dropout,
    GlobalAveragePooling1D
)

In [5]:
file_path = "/content/drive/MyDrive/cryptocast/data/processed/bitcoin_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (4999, 8)


,Date,Price,Open,High,Low,Vol.,Change %,Vol_Log
0,2010-07-18,0.1,0.0,0.1,0.1,80.0,0.0,4.394449
1,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0,6.347389
2,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0,5.564520
3,2010-07-21,0.1,0.1,0.1,0.1,580.0,0.0,6.364751
4,2010-07-22,0.1,0.1,0.1,0.1,2160.0,0.0,7.678326


In [6]:
df['Date'] = pd.to_datetime(df['Date'])

print(df['Date'].dtype)
print(df['Date'].min())
print(df['Date'].max())

datetime64[ns]
2010-07-18 00:00:00
2024-03-24 00:00:00


In [7]:
features = [
    'Price',
    'Open',
    'High',
    'Low',
    'Vol_Log',
    'Change %'
]

target = 'Price'

X = df[features]
y = df[target]

print("Features:", features)
print("Target:", target)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['Price', 'Open', 'High', 'Low', 'Vol_Log', 'Change %']
Target: Price
X shape: (4999, 6)
y shape: (4999,)


In [8]:
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:")
print(train_df['Date'].min(), "to", train_df['Date'].max())

print("\nTesting period:")
print(test_df['Date'].min(), "to", test_df['Date'].max())

Training shape: (3999, 8)
Testing shape: (1000, 8)

Training period:
2010-07-18 00:00:00 to 2021-06-28 00:00:00

Testing period:
2021-06-29 00:00:00 to 2024-03-24 00:00:00


In [9]:
X_train = train_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (3999, 6)
X_test: (1000, 6)
y_train: (3999,)
y_test: (1000,)


In [10]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = X_train_scaled[:, 0]
y_test_scaled = X_test_scaled[:, 0]

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("y_train_scaled shape:", y_train_scaled.shape)
print("y_test_scaled shape:", y_test_scaled.shape)

X_train_scaled shape: (3999, 6)
X_test_scaled shape: (1000, 6)
y_train_scaled shape: (3999,)
y_test_scaled shape: (1000,)


In [11]:
window_size = 60

def create_sequences(data, target, window_size, horizon):
    X_seq = []
    y_seq = []

    for i in range(window_size, len(data) - horizon + 1):
        X_seq.append(data[i-window_size:i])
        y_seq.append(target[i:i+horizon])

    return np.array(X_seq), np.array(y_seq)

In [12]:
X_train_seq_1d, y_train_seq_1d = create_sequences(
    X_train_scaled,
    y_train_scaled,
    window_size,
    1
)

print("X shape:", X_train_seq_1d.shape)
print("y shape:", y_train_seq_1d.shape)

X shape: (3939, 60, 6)
y shape: (3939, 1)


In [13]:
val_size = int(len(X_train_seq_1d) * 0.10)

X_train_1d = X_train_seq_1d[:-val_size]
y_train_1d = y_train_seq_1d[:-val_size]

X_val_1d = X_train_seq_1d[-val_size:]
y_val_1d = y_train_seq_1d[-val_size:]

print("Training:", X_train_1d.shape, y_train_1d.shape)
print("Validation:", X_val_1d.shape, y_val_1d.shape)

Training: (3546, 60, 6) (3546, 1)
Validation: (393, 60, 6) (393, 1)


In [14]:
def build_transformer_model(
    input_shape,
    output_units,
    num_heads=4,
    key_dim=32,
    ff_dim=64,
    dropout=0.1
):
    inputs = Input(shape=input_shape)

    # Multi-Head Self-Attention
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(inputs, inputs)

    attention_output = Dropout(dropout)(attention_output)

    # Residual connection + normalization
    x = LayerNormalization(epsilon=1e-6)(
        inputs + attention_output
    )

    # Feed-forward network
    ff_output = Dense(ff_dim, activation='relu')(x)
    ff_output = Dense(input_shape[-1])(ff_output)

    ff_output = Dropout(dropout)(ff_output)

    # Residual connection + normalization
    x = LayerNormalization(epsilon=1e-6)(
        x + ff_output
    )

    # Convert sequence representation to output
    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation='relu')(x)

    outputs = Dense(output_units)(x)

    model = Model(inputs, outputs)

    return model


transformer_1d = build_transformer_model(
    input_shape=(60, 6),
    output_units=1
)

transformer_1d.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

transformer_1d.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 6)     │      3,462 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 60, 6)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 60, 6)     │          0 │ input_layer[0][0… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 60, 6)     │         12 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 60, 64)    │        448 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 60, 6)     │        390 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 60, 6)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 60, 6)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 6)     │         12 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 6)         │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │        448 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,837 (18.89 KB)

 Trainable params: 4,837 (18.89 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history_transformer_1d = transformer_1d.fit(
    X_train_1d,
    y_train_1d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_1d, y_val_1d),
    shuffle=False
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - loss: 0.0025 - mae: 0.0342 - val_loss: 0.1542 - val_mae: 0.2827
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 0.0032 - mae: 0.0389 - val_loss: 0.1367 - val_mae: 0.2666
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0041 - mae: 0.0445 - val_loss: 0.1524 - val_mae: 0.2773
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 73ms/step - loss: 0.0036 - mae: 0.0407 - val_loss: 0.1582 - val_mae: 0.2860
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - loss: 0.0023 - mae: 0.0304 - val_loss: 0.1849 - val_mae: 0.3177
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - loss: 0.0028 - mae: 0.0366 - val_loss: 0.1470 - val_mae: 0.2730
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0035 - mae: 0.0405 - val_loss: 0.1674 - val_mae: 0.2945
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - loss: 0.0031 - mae: 0.0372 - val_loss: 0.1510 - val_mae: 0.2766
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/

In [16]:
X_test_context_1d = np.concatenate(
    [X_train_scaled[-window_size:], X_test_scaled],
    axis=0
)

y_test_context_1d = np.concatenate(
    [y_train_scaled[-window_size:], y_test_scaled],
    axis=0
)

X_test_seq_1d, y_test_seq_1d = create_sequences(
    X_test_context_1d,
    y_test_context_1d,
    window_size,
    1
)

print("X_test_seq_1d shape:", X_test_seq_1d.shape)
print("y_test_seq_1d shape:", y_test_seq_1d.shape)

X_test_seq_1d shape: (1000, 60, 6)
y_test_seq_1d shape: (1000, 1)


In [17]:
predictions_1d = transformer_1d.predict(X_test_seq_1d)

print("Predictions shape:", predictions_1d.shape)
print("Prediction range:", predictions_1d.min(), "to", predictions_1d.max())

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
Predictions shape: (1000, 1)
Prediction range: 0.15900116 to 0.20158498


In [18]:
price_min = scaler.data_min_[0]
price_max = scaler.data_max_[0]

predictions_1d_original = (
    predictions_1d * (price_max - price_min)
) + price_min

actuals_1d_original = (
    y_test_seq_1d * (price_max - price_min)
) + price_min

print("Predicted price range:",
      predictions_1d_original.min(),
      "to",
      predictions_1d_original.max())

print("Actual price range:",
      actuals_1d_original.min(),
      "to",
      actuals_1d_original.max())

Predicted price range: 10103.160696864128 to 12808.970890307428
Actual price range: 15776.2 to 73066.3


In [19]:
print("Scaled prediction range:")
print(predictions_1d.min(), "to", predictions_1d.max())

print("\nFirst 10 scaled predictions:")
print(predictions_1d[:10].flatten())

print("\nFirst 10 actual scaled values:")
print(y_test_seq_1d[:10].flatten())

Scaled prediction range:
0.15900116 to 0.20158498

First 10 scaled predictions:
[0.20110378 0.20107818 0.20093241 0.20074324 0.20050593 0.20029478
 0.20011707 0.19988039 0.19963036 0.19937517]

First 10 actual scaled values:
[0.56396205 0.55124896 0.52790491 0.532151   0.54677782 0.55551866
 0.53017431 0.53863817 0.53300714 0.51724561]


In [20]:
print("Train Price scaled range:")
print(y_train_scaled.min(), "to", y_train_scaled.max())

print("\nTest Price scaled range:")
print(y_test_scaled.min(), "to", y_test_scaled.max())

print("\nTrain Price actual range:")
print(y_train.min(), "to", y_train.max())

print("\nTest Price actual range:")
print(y_test.min(), "to", y_test.max())

Train Price scaled range:
0.0 to 0.9999999999999999

Test Price scaled range:
0.2482829929745927 to 1.1499099791000427

Train Price actual range:
0.1 to 63540.9

Test Price actual range:
15776.2 to 73066.3


In [21]:
print("Prediction mean:", predictions_1d.mean())
print("Prediction std:", predictions_1d.std())

print("\nActual mean:", y_test_seq_1d.mean())
print("Actual std:", y_test_seq_1d.std())

Prediction mean: 0.18405615
Prediction std: 0.013194674

Actual mean: 0.5394628380505123
Actual std: 0.2030712654921018


In [22]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, d_model):
        super().__init__()

        positions = np.arange(sequence_length)[:, np.newaxis]
        dimensions = np.arange(d_model)[np.newaxis, :]

        angle_rates = 1 / np.power(
            10000,
            (2 * (dimensions // 2)) / np.float32(d_model)
        )

        angle_rads = positions * angle_rates

        pos_encoding = np.zeros((sequence_length, d_model))

        pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.cast(
            pos_encoding[np.newaxis, ...],
            dtype=tf.float32
        )

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]


In [23]:
def build_transformer_model_v2(
    input_shape,
    output_units,
    num_heads=4,
    key_dim=32,
    ff_dim=128,
    dropout=0.1
):
    inputs = Input(shape=input_shape)

    # Project input features to a richer representation
    x = Dense(64)(inputs)

    # Add positional information
    x = PositionalEncoding(
        sequence_length=input_shape[0],
        d_model=64
    )(x)

    # Multi-Head Attention
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(x, x)

    attention_output = Dropout(dropout)(attention_output)

    # Residual connection + normalization
    x = LayerNormalization(epsilon=1e-6)(
        x + attention_output
    )

    # Feed-forward network
    ff_output = Dense(ff_dim, activation='relu')(x)
    ff_output = Dense(64)(ff_output)
    ff_output = Dropout(dropout)(ff_output)

    # Residual connection + normalization
    x = LayerNormalization(epsilon=1e-6)(
        x + ff_output
    )

    # Convert sequence representation to prediction
    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout)(x)

    outputs = Dense(output_units)(x)

    model = Model(inputs, outputs)

    return model


In [24]:
transformer_1d_v2 = build_transformer_model_v2(
    input_shape=(60, 6),
    output_units=1
)

transformer_1d_v2.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

transformer_1d_v2.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 60, 64)    │        448 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 60, 64)    │          0 │ dense_4[0][0]     │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     33,216 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 60, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 60, 64)    │      8,256 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 60, 64)    │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         65 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,721 (213.75 KB)

 Trainable params: 54,721 (213.75 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history_transformer_1d_v2 = transformer_1d_v2.fit(
    X_train_1d,
    y_train_1d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_1d, y_val_1d),
    shuffle=False
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - loss: 0.0501 - mae: 0.1316 - val_loss: 0.1922 - val_mae: 0.3541
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - loss: 0.0021 - mae: 0.0315 - val_loss: 0.1859 - val_mae: 0.3415
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 68ms/step - loss: 0.0020 - mae: 0.0275 - val_loss: 0.1785 - val_mae: 0.3000
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step - loss: 0.0035 - mae: 0.0395 - val_loss: 0.1261 - val_mae: 0.2575
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - loss: 0.0034 - mae: 0.0352 - val_loss: 0.2079 - val_mae: 0.3464
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - loss: 0.0023 - mae: 0.0297 - val_loss: 0.1809 - val_mae: 0.3251
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - loss: 0.0036 - mae: 0.0411 - val_loss: 0.2322 - val_mae: 0.3894
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step - loss: 0.0036 - mae: 0.0478 - val_loss: 0.2359 - val_mae: 0.3936
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms

In [26]:
print("Training target range:")
print(y_train_1d.min(), "to", y_train_1d.max())

print("\nValidation target range:")
print(y_val_1d.min(), "to", y_val_1d.max())

print("\nTraining target mean:", y_train_1d.mean())
print("Validation target mean:", y_val_1d.mean())

Training target range:
0.0 to 0.30445634930627247

Validation target range:
0.1417703270969204 to 0.9999999999999999

Training target mean: 0.03874996815135057
Validation target mean: 0.4421526015638029


In [27]:
train_end_date = train_df['Date'].iloc[-1]

val_start_index = len(train_df) - val_size

val_start_date = train_df['Date'].iloc[val_start_index]

print("Training data ends:", train_end_date)
print("Validation starts:", val_start_date)
print("Validation ends:", train_df['Date'].iloc[-1])

Training data ends: 2021-06-28 00:00:00
Validation starts: 2020-06-01 00:00:00
Validation ends: 2021-06-28 00:00:00


In [28]:
def build_transformer_model_v3(
    input_shape,
    output_units,
    num_heads=4,
    key_dim=16,
    ff_dim=128,
    dropout=0.1
):
    inputs = Input(shape=input_shape)

    x = Dense(64)(inputs)

    x = PositionalEncoding(
        sequence_length=input_shape[0],
        d_model=64
    )(x)

    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(x, x)

    attention_output = Dropout(dropout)(attention_output)

    x = LayerNormalization(epsilon=1e-6)(
        x + attention_output
    )

    ff_output = Dense(ff_dim, activation='relu')(x)
    ff_output = Dense(64)(ff_output)
    ff_output = Dropout(dropout)(ff_output)

    x = LayerNormalization(epsilon=1e-6)(
        x + ff_output
    )

    # Use the most recent timestep
    x = x[:, -1, :]

    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout)(x)

    outputs = Dense(output_units)(x)

    return Model(inputs, outputs)

In [29]:
def build_transformer_model_v3(
    input_shape,
    output_units,
    num_heads=4,
    key_dim=16,
    ff_dim=128,
    dropout=0.1
):
    inputs = Input(shape=input_shape)

    x = Dense(64)(inputs)

    x = PositionalEncoding(
        sequence_length=input_shape[0],
        d_model=64
    )(x)

    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(x, x)

    attention_output = Dropout(dropout)(attention_output)

    x = LayerNormalization(epsilon=1e-6)(
        x + attention_output
    )

    ff_output = Dense(ff_dim, activation='relu')(x)
    ff_output = Dense(64)(ff_output)
    ff_output = Dropout(dropout)(ff_output)

    x = LayerNormalization(epsilon=1e-6)(
        x + ff_output
    )

    # Use the most recent timestep
    x = x[:, -1, :]

    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout)(x)

    outputs = Dense(output_units)(x)

    return Model(inputs, outputs)

In [30]:
transformer_1d_v3 = build_transformer_model_v3(
    input_shape=(60, 6),
    output_units=1
)

transformer_1d_v3.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

transformer_1d_v3.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 60, 64)    │        448 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 60, 64)    │          0 │ dense_9[0][0]     │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 60, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 60, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 60, 64)    │      8,256 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 60, 64)    │          0 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 64)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │      4,160 │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 64)        │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1)         │         65 │ dropout_10[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,145 (149.00 KB)

 Trainable params: 38,145 (149.00 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history_transformer_1d_v3 = transformer_1d_v3.fit(
    X_train_1d,
    y_train_1d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_1d, y_val_1d),
    shuffle=False
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - loss: 0.0658 - mae: 0.1743 - val_loss: 0.2293 - val_mae: 0.3610
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0048 - mae: 0.0520 - val_loss: 0.2129 - val_mae: 0.3595
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 54ms/step - loss: 0.0034 - mae: 0.0381 - val_loss: 0.1760 - val_mae: 0.3122
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 0.0040 - mae: 0.0411 - val_loss: 0.2040 - val_mae: 0.3437
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0034 - mae: 0.0404 - val_loss: 0.1849 - val_mae: 0.3264
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - loss: 0.0042 - mae: 0.0472 - val_loss: 0.1614 - val_mae: 0.3063
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0035 - mae: 0.0453 - val_loss: 0.2344 - val_mae: 0.3929
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - loss: 0.0036 - mae: 0.0490 - val_loss: 0.2333 - val_mae: 0.3914
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/

In [32]:
history_transformer_1d_v3 = transformer_1d_v3.fit(
    X_train_1d,
    y_train_1d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_1d, y_val_1d),
    shuffle=False
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0037 - mae: 0.0497 - val_loss: 0.2264 - val_mae: 0.3842
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 0.0037 - mae: 0.0497 - val_loss: 0.2299 - val_mae: 0.3876
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - loss: 0.0037 - mae: 0.0496 - val_loss: 0.2194 - val_mae: 0.3758
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0038 - mae: 0.0500 - val_loss: 0.2310 - val_mae: 0.3884
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - loss: 0.0037 - mae: 0.0497 - val_loss: 0.2307 - val_mae: 0.3881
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0037 - mae: 0.0498 - val_loss: 0.2307 - val_mae: 0.3881
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - loss: 0.0037 - mae: 0.0498 - val_loss: 0.2306 - val_mae: 0.3880
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - loss: 0.0037 - mae: 0.0498 - val_loss: 0.2304 - val_mae: 0.3879
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms

In [33]:
naive_val_predictions = X_val_1d[:, -1, 0].reshape(-1, 1)

naive_val_mae = mean_absolute_error(
    y_val_1d,
    naive_val_predictions
)

naive_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_1d,
        naive_val_predictions
    )
)

naive_val_mape = np.mean(
    np.abs(
        (y_val_1d - naive_val_predictions)
        / y_val_1d
    )
) * 100

print("Naive Validation MAE:", naive_val_mae)
print("Naive Validation RMSE:", naive_val_rmse)
print("Naive Validation MAPE:", naive_val_mape)

Naive Validation MAE: 0.01386554364076242
Naive Validation RMSE: 0.023216768278201832
Naive Validation MAPE: 2.730266459364907


In [34]:
df['Price_Return'] = np.log(
    df['Price'] / df['Price'].shift(1)
)

df['Price_Return'] = df['Price_Return'].fillna(0)

print(df[['Date', 'Price', 'Price_Return']].head())
print("\nReturn statistics:")
print(df['Price_Return'].describe())

        Date  Price  Price_Return
0 2010-07-18    0.1           0.0
1 2010-07-19    0.1           0.0
2 2010-07-20    0.1           0.0
3 2010-07-21    0.1           0.0
4 2010-07-22    0.1           0.0

Return statistics:
count    4999.000000
mean        0.002684
std         0.061232
min        -0.848829
25%        -0.012303
50%         0.000000
75%         0.018206
max         1.474180
Name: Price_Return, dtype: float64


In [35]:
features_return = [
    'Price',
    'Open',
    'High',
    'Low',
    'Vol_Log',
    'Change %'
]

target_return = 'Price_Return'

X_return = df[features_return]
y_return = df[target_return]

print("Features shape:", X_return.shape)
print("Target shape:", y_return.shape)

Features shape: (4999, 6)
Target shape: (4999,)


In [36]:
split_index = int(len(df) * 0.80)

train_df_return = df.iloc[:split_index].copy()
test_df_return = df.iloc[split_index:].copy()

X_train_return = train_df_return[features_return]
X_test_return = test_df_return[features_return]

y_train_return = train_df_return[target_return]
y_test_return = test_df_return[target_return]

print("Train shape:", train_df_return.shape)
print("Test shape:", test_df_return.shape)

print("\nTrain period:")
print(train_df_return['Date'].min(), "to", train_df_return['Date'].max())

print("\nTest period:")
print(test_df_return['Date'].min(), "to", test_df_return['Date'].max())

Train shape: (3999, 9)
Test shape: (1000, 9)

Train period:
2010-07-18 00:00:00 to 2021-06-28 00:00:00

Test period:
2021-06-29 00:00:00 to 2024-03-24 00:00:00


In [37]:
scaler_return = MinMaxScaler()

X_train_return_scaled = scaler_return.fit_transform(X_train_return)
X_test_return_scaled = scaler_return.transform(X_test_return)

y_train_return_scaled = y_train_return.values
y_test_return_scaled = y_test_return.values

print("X_train scaled shape:", X_train_return_scaled.shape)
print("X_test scaled shape:", X_test_return_scaled.shape)

print("\nReturn target range:")
print(y_train_return_scaled.min(), "to", y_train_return_scaled.max())

X_train scaled shape: (3999, 6)
X_test scaled shape: (1000, 6)

Return target range:
-0.848828668187098 to 1.474179803720085


In [38]:
window_size = 60

X_train_seq_return, y_train_seq_return = create_sequences(
    X_train_return_scaled,
    y_train_return_scaled,
    window_size,
    1
)

print("X_train_seq_return shape:", X_train_seq_return.shape)
print("y_train_seq_return shape:", y_train_seq_return.shape)

X_train_seq_return shape: (3939, 60, 6)
y_train_seq_return shape: (3939, 1)


In [39]:
val_size = int(len(X_train_seq_return) * 0.10)

X_train_return_final = X_train_seq_return[:-val_size]
y_train_return_final = y_train_seq_return[:-val_size]

X_val_return = X_train_seq_return[-val_size:]
y_val_return = y_train_seq_return[-val_size:]

print("Training shape:", X_train_return_final.shape)
print("Validation shape:", X_val_return.shape)
print("Training target shape:", y_train_return_final.shape)
print("Validation target shape:", y_val_return.shape)

Training shape: (3546, 60, 6)
Validation shape: (393, 60, 6)
Training target shape: (3546, 1)
Validation target shape: (393, 1)


In [40]:
transformer_return = build_transformer_model_v3(
    input_shape=(60, 6),
    output_units=1
)

transformer_return.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

transformer_return.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 60, 64)    │        448 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 60, 64)    │          0 │ dense_14[0][0]    │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 60, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 60, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 60, 64)    │      8,256 │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 60, 64)    │          0 │ dense_16[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 64)        │          0 │ layer_normalizat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 64)        │      4,160 │ get_item_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 64)        │          0 │ dense_17[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 1)         │         65 │ dropout_14[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,145 (149.00 KB)

 Trainable params: 38,145 (149.00 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
history_transformer_return = transformer_return.fit(
    X_train_return_final,
    y_train_return_final,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_return, y_val_return),
    shuffle=False
)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 13s 77ms/step - loss: 0.0460 - mae: 0.1314 - val_loss: 0.0230 - val_mae: 0.1029
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0062 - mae: 0.0484 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - loss: 0.0052 - mae: 0.0376 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.0049 - mae: 0.0348 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - loss: 0.0049 - mae: 0.0344 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - loss: 0.0049 - mae: 0.0344 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0049 - mae: 0.0341 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - loss: 0.0049 - mae: 0.0340 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms

In [42]:
X_test_context_return = np.concatenate(
    [X_train_return_scaled[-window_size:], X_test_return_scaled],
    axis=0
)

y_test_context_return = np.concatenate(
    [y_train_return_scaled[-window_size:], y_test_return_scaled],
    axis=0
)

X_test_seq_return, y_test_seq_return = create_sequences(
    X_test_context_return,
    y_test_context_return,
    window_size,
    1
)

print("X_test_seq_return shape:", X_test_seq_return.shape)
print("y_test_seq_return shape:", y_test_seq_return.shape)

X_test_seq_return shape: (1000, 60, 6)
y_test_seq_return shape: (1000, 1)


In [43]:
predicted_returns_1d = transformer_return.predict(
    X_test_seq_return
)

print("Predicted returns shape:", predicted_returns_1d.shape)

print(
    "Predicted return range:",
    predicted_returns_1d.min(),
    "to",
    predicted_returns_1d.max()
)

print("\nFirst 10 predicted returns:")
print(predicted_returns_1d[:10].flatten())

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Predicted returns shape: (1000, 1)
Predicted return range: -0.004948556 to 0.0013670714

First 10 predicted returns:
[0.00106319 0.00105312 0.00105639 0.00106938 0.00107528 0.00106459
 0.00105194 0.00107119 0.00106907 0.00106872]


In [44]:
previous_prices_1d = X_test_context_return[window_size - 1:-1, 0]

predicted_prices_1d = (
    previous_prices_1d *
    np.exp(predicted_returns_1d.flatten())
)

actual_prices_1d = test_df_return['Price'].values

print("Predicted price range:",
      predicted_prices_1d.min(),
      "to",
      predicted_prices_1d.max())

print("Actual price range:",
      actual_prices_1d.min(),
      "to",
      actual_prices_1d.max())

Predicted price range: 0.2486226554820524 to 1.1442336353819247
Actual price range: 15776.2 to 73066.3


In [45]:
mae_transformer_1d = mean_absolute_error(
    actual_prices_1d,
    predicted_prices_1d
)

rmse_transformer_1d = np.sqrt(
    mean_squared_error(
        actual_prices_1d,
        predicted_prices_1d
    )
)

mape_transformer_1d = np.mean(
    np.abs(
        (actual_prices_1d - predicted_prices_1d)
        / actual_prices_1d
    )
) * 100

print("Transformer 1-Day Results")
print("-------------------------")
print("MAE :", mae_transformer_1d)
print("RMSE:", rmse_transformer_1d)
print("MAPE:", mape_transformer_1d, "%")

Transformer 1-Day Results
-------------------------
MAE : 34277.46081075941
RMSE: 36625.5995415474
MAPE: 99.99842484046158 %


In [46]:
debug_df = pd.DataFrame({
    'Previous_Price': previous_prices_1d[:10],
    'Predicted_Return': predicted_returns_1d.flatten()[:10],
    'Predicted_Price': predicted_prices_1d[:10],
    'Actual_Price': actual_prices_1d[:10]
})

debug_df

,Previous_Price,Predicted_Return,Predicted_Price,Actual_Price
0,0.542577,0.001063,0.543155,35834.7
1,0.563962,0.001053,0.564556,35026.9
2,0.551249,0.001056,0.551832,33543.6
3,0.527905,0.001069,0.528470,33813.4
4,0.532151,0.001075,0.532724,34742.8
5,0.546778,0.001065,0.547360,35298.2
6,0.555519,0.001052,0.556103,33687.8
7,0.530174,0.001071,0.530743,34225.6
8,0.538638,0.001069,0.539214,33867.8
9,0.533007,0.001069,0.533577,32866.3


In [47]:
price_min_return = scaler_return.data_min_[0]
price_max_return = scaler_return.data_max_[0]

previous_prices_1d_original = (
    previous_prices_1d *
    (price_max_return - price_min_return)
) + price_min_return

predicted_prices_1d = (
    previous_prices_1d_original *
    np.exp(predicted_returns_1d.flatten())
)

actual_prices_1d = test_df_return['Price'].values

print("Previous actual price range:",
      previous_prices_1d_original.min(),
      "to",
      previous_prices_1d_original.max())

print("\nPredicted price range:",
      predicted_prices_1d.min(),
      "to",
      predicted_prices_1d.max())

print("\nActual price range:",
      actual_prices_1d.min(),
      "to",
      actual_prices_1d.max())

Previous actual price range: 15776.2 to 73066.3

Predicted price range: 15797.782564258576 to 72705.62008544207

Actual price range: 15776.2 to 73066.3


In [48]:
# ============================================================
# TRANSFORMER 3-DAY + 7-DAY MODEL GENERATION
# ============================================================

import os
import joblib
import numpy as np
import tensorflow as tf

MODEL_DIR = "/content/drive/MyDrive/cryptocast/streamlit/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3-DAY TRANSFORMER
# ------------------------------------------------------------

X_train_3d, y_train_3d = create_sequences(
    X_train_scaled,
    y_train_scaled,
    window_size=60,
    horizon=3
)

print("3-Day:")
print("X:", X_train_3d.shape)
print("y:", y_train_3d.shape)

val_index_3d = int(len(X_train_3d) * 0.90)

X_train_final_3d = X_train_3d[:val_index_3d]
y_train_final_3d = y_train_3d[:val_index_3d]

X_val_3d = X_train_3d[val_index_3d:]
y_val_3d = y_train_3d[val_index_3d:]

transformer_3d = build_transformer_model_v3(
    input_shape=(60, 6),
    output_units=3
)

transformer_3d.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

transformer_3d.summary()

history_transformer_3d = transformer_3d.fit(
    X_train_final_3d,
    y_train_final_3d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_3d, y_val_3d),
    shuffle=False
)

# IMPORTANT CHECK
print("3-Day model output shape:", transformer_3d.output_shape)

test_prediction_3d = transformer_3d.predict(
    X_val_3d[:1],
    verbose=0
)

print("3-Day prediction shape:", test_prediction_3d.shape)

# Save
transformer_3d.save(
    os.path.join(MODEL_DIR, "transformer_3d.keras")
)

print("✅ transformer_3d.keras saved")

3-Day:
X: (3937, 60, 6)
y: (3937, 3)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 60, 64)    │        448 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 60, 64)    │          0 │ dense_19[0][0]    │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 60, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 60, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 60, 64)    │      8,256 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 60, 64)    │          0 │ dense_21[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 64)        │          0 │ layer_normalizat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 64)        │      4,160 │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 64)        │          0 │ dense_22[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 3)         │        195 │ dropout_18[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,275 (149.51 KB)

 Trainable params: 38,275 (149.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - loss: 0.0468 - mae: 0.1429 - val_loss: 0.1657 - val_mae: 0.3025
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 80ms/step - loss: 0.0045 - mae: 0.0432 - val_loss: 0.1751 - val_mae: 0.3287
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - loss: 0.0036 - mae: 0.0391 - val_loss: 0.2201 - val_mae: 0.3836
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 0.0036 - mae: 0.0467 - val_loss: 0.2334 - val_mae: 0.3912
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0036 - mae: 0.0489 - val_loss: 0.2313 - val_mae: 0.3891
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - loss: 0.0037 - mae: 0.0494 - val_loss: 0.2302 - val_mae: 0.3880
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 0.0037 - mae: 0.0495 - val_loss: 0.2199 - val_mae: 0.3808
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 0.0037 - mae: 0.0496 - val_loss: 0.2274 - val_mae: 0.3859
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms

In [49]:
# ============================================================
# TRANSFORMER 7-DAY MODEL
# ============================================================

X_train_7d, y_train_7d = create_sequences(
    X_train_scaled,
    y_train_scaled,
    window_size=60,
    horizon=7
)

print("7-Day:")
print("X:", X_train_7d.shape)
print("y:", y_train_7d.shape)

val_index_7d = int(len(X_train_7d) * 0.90)

X_train_final_7d = X_train_7d[:val_index_7d]
y_train_final_7d = y_train_7d[:val_index_7d]

X_val_7d = X_train_7d[val_index_7d:]
y_val_7d = y_train_7d[val_index_7d:]

transformer_7d = build_transformer_model_v3(
    input_shape=(60, 6),
    output_units=7
)

transformer_7d.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

transformer_7d.summary()

history_transformer_7d = transformer_7d.fit(
    X_train_final_7d,
    y_train_final_7d,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_7d, y_val_7d),
    shuffle=False
)

# IMPORTANT CHECK
print("7-Day model output shape:", transformer_7d.output_shape)

test_prediction_7d = transformer_7d.predict(
    X_val_7d[:1],
    verbose=0
)

print("7-Day prediction shape:", test_prediction_7d.shape)

# Save
transformer_7d.save(
    os.path.join(MODEL_DIR, "transformer_7d.keras")
)

print("✅ transformer_7d.keras saved")

7-Day:
X: (3933, 60, 6)
y: (3933, 7)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 60, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 60, 64)    │        448 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 60, 64)    │          0 │ dense_24[0][0]    │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 60, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 60, 64)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_20[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 60, 64)    │      8,256 │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 60, 64)    │          0 │ dense_26[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 60, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_21[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 64)        │          0 │ layer_normalizat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 64)        │      4,160 │ get_item_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_22          │ (None, 64)        │          0 │ dense_27[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 7)         │        455 │ dropout_22[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,535 (150.53 KB)

 Trainable params: 38,535 (150.53 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 13s 71ms/step - loss: 0.0285 - mae: 0.0977 - val_loss: 0.1434 - val_mae: 0.2872
Epoch 2/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - loss: 0.0038 - mae: 0.0405 - val_loss: 0.1797 - val_mae: 0.3433
Epoch 3/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - loss: 0.0035 - mae: 0.0456 - val_loss: 0.1779 - val_mae: 0.3320
Epoch 4/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 0.0035 - mae: 0.0473 - val_loss: 0.1376 - val_mae: 0.2653
Epoch 5/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - loss: 0.0036 - mae: 0.0481 - val_loss: 0.1451 - val_mae: 0.2905
Epoch 6/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - loss: 0.0035 - mae: 0.0476 - val_loss: 0.1367 - val_mae: 0.2897
Epoch 7/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - loss: 0.0035 - mae: 0.0470 - val_loss: 0.1276 - val_mae: 0.2751
Epoch 8/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 0.0035 - mae: 0.0468 - val_loss: 0.1119 - val_mae: 0.2571
Epoch 9/20
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms

In [50]:
print(os.listdir(MODEL_DIR))

['transformer_1d.keras', 'transformer_3d.keras', 'transformer_7d.keras', 'scaler_1d.pkl', 'scaler_3d.pkl', 'scaler_7d.pkl', 'cnn_3d.keras', 'cnn_7d.keras', 'cnn_scaler_3d.pkl', 'cnn_scaler_7d.pkl', 'rnn_1d.keras', 'rnn_3d.keras', 'rnn_7d.keras', 'rnn_scaler_1d.pkl', 'rnn_scaler_3d.pkl', 'rnn_scaler_7d.pkl', 'lstm_1d.keras', 'lstm_3d.keras', 'lstm_7d.keras', 'lstm_scaler_1d.pkl', 'lstm_scaler_3d.pkl', 'lstm_scaler_7d.pkl', 'cnn_1d.keras', 'cnn_scaler.pkl']


In [52]:
import os

for root, dirs, files in os.walk("/content/mydrive/MyDrive"):
    if "transformer_1d.keras" in files:
        print("FOUND:", os.path.join(root, "transformer_1d.keras"))

In [53]:
import os

print("mydrive exists:", os.path.exists("/content/mydrive/MyDrive"))
print("drive exists:", os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive/MyDrive"):
    print("drive model search:")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        if "transformer_1d.keras" in files:
            print(os.path.join(root, "transformer_1d.keras"))

mydrive exists: False
drive exists: True
drive model search:
/content/drive/MyDrive/cryptocast/streamlit/models/transformer_1d.keras


In [55]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, sequence_length=60, d_model=64, **kwargs):
        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.d_model = d_model

    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]
        depth = tf.shape(inputs)[2]

        positions = tf.cast(
            tf.range(seq_len)[:, None],
            tf.float32
        )

        dimensions = tf.cast(
            tf.range(depth)[None, :],
            tf.float32
        )

        rates = tf.pow(
            10000.0,
            -(2.0 * tf.floor(dimensions / 2.0))
            / tf.cast(depth, tf.float32)
        )

        angles = positions * rates

        even = tf.equal(
            tf.math.floormod(tf.range(depth), 2),
            0
        )[None, :]

        encoding = tf.where(
            even,
            tf.sin(angles),
            tf.cos(angles)
        )

        return inputs + encoding[None, :, :]

    def get_config(self):

        config = super().get_config()

        config.update({
            "sequence_length": self.sequence_length,
            "d_model": self.d_model
        })

        return config

In [56]:
from tensorflow.keras.models import load_model

MODEL_DIR = "/content/drive/MyDrive/cryptocast/streamlit/models"

for name in [
    "transformer_1d.keras",
    "transformer_3d.keras",
    "transformer_7d.keras"
]:

    model = load_model(
        f"{MODEL_DIR}/{name}",
        custom_objects={
            "PositionalEncoding": PositionalEncoding
        },
        compile=False
    )

    print(name, "->", model.output_shape)

transformer_1d.keras -> (None, 1)
transformer_3d.keras -> (None, 3)
transformer_7d.keras -> (None, 7)


In [58]:
import os
import joblib
import numpy as np
from tensorflow.keras.models import load_model

MODEL_DIR = "/content/drive/MyDrive/cryptocast/streamlit/models"

model = load_model(
    os.path.join(MODEL_DIR, "transformer_3d.keras"),
    custom_objects={"PositionalEncoding": PositionalEncoding},
    compile=False
)

scaler = joblib.load(
    os.path.join(MODEL_DIR, "scaler_3d.pkl")
)

print("Model output:", model.output_shape)
print("Scaler:", type(scaler))
print("Scaler features:", scaler.n_features_in_)

Model output: (None, 3)
Scaler: <class 'sklearn.preprocessing._data.MinMaxScaler'>
Scaler features: 6
